In [7]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

In [8]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [4]:
# set seed
random.seed(42)

# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

# # usage
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [11]:
from itertools import product
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

def grid_search(
    X,
    y,
    param_grid,
    verbose=True,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    return_details=True,
):
    """
    Parallel grid search.

    Parameters
    ----------
    return_details : bool, default True
        If False, skip collecting and concatenating window-level details
        (MUCH faster and lower memory usage).

    Returns
    -------
    results_df : pd.DataFrame
        Summary metrics for each configuration.
    coefficients_df : pd.DataFrame or None
        Detailed coefficients if return_details=True, else None.
    """

    combos = list(product(
        param_grid["window_sizes"],
        param_grid["n_lags"],
        param_grid["lambdas"],
    ))

    if verbose:
        print(f"Testing {len(combos)} configurations...")

    # -------------------------------------------------
    # Wrapper to prevent single failure from killing run
    # -------------------------------------------------
    def _safe_run(args):
        w, L, lam = args
        try:
            return estimate_single_config(X, y, w, L, lam)
        except Exception as e:
            if verbose:
                print(f"❌ Failed config (w={w}, L={L}, λ={lam}): {e}")
            return None

    iterator = tqdm(combos, desc="Grid search") if verbose else combos

    # -------------------------------------------------
    # Parallel execution
    # -------------------------------------------------
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(_safe_run)(args) for args in iterator
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    summary_list = []
    details_list = [] if return_details else None

    for res in results:
        if res is None:
            continue

        summary_list.append(res.get("summary", {}))

        if return_details:
            det = res.get("details", None)
            if det is not None and not det.empty:
                details_list.append(det)

    # -------------------------------------------------
    # Build summary DataFrame
    # -------------------------------------------------
    results_df = pd.DataFrame(summary_list)

    if not results_df.empty and "r2_oos_stage2" in results_df.columns:
        results_df = results_df.sort_values(
            "r2_oos_stage2", ascending=False
        ).reset_index(drop=True)

    # -------------------------------------------------
    # Build details DataFrame (optional)
    # -------------------------------------------------
    coefficients_df = None
    if return_details and details_list:
        coefficients_df = pd.concat(details_list, ignore_index=True)

    # -------------------------------------------------
    # Reporting
    # -------------------------------------------------
    if verbose:
        print("\n" + "=" * 80)
        print("GRID SEARCH COMPLETE")
        print("=" * 80)

        if "kappa" in results_df.columns:
            n_failed = results_df["kappa"].isna().sum()
            if n_failed > 0:
                print(f"⚠️  {n_failed}/{len(results_df)} configurations failed")

    return results_df, coefficients_df


## Grid Search

In [5]:
import numpy as np
import pandas as pd

# -------------------------------------------------
# Objective function: maximize R^2 only
# -------------------------------------------------
def objective_function(row, r2_col="r2_insample_stage2"):
    r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
    if not np.isfinite(r2):
        return -1.0
    return r2


# -------------------------------------------------
# Iterative grid refinement
# -------------------------------------------------
summary_df, details_df, best_overall


Testing 12 configurations...


Grid search: 100%|██████████| 12/12 [00:00<00:00, 15.78it/s]



GRID SEARCH COMPLETE
window_size    78.000000
n_lags          1.000000
lambda          0.000100
objective       0.000042
Name: 8, dtype: float64


/var/folders/hb/bc0srnt52zx6lbw4t2k_t_k80000gn/T/ipykernel_97527/2362108959.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coefficients_df = pd.concat(details_list, ignore_index=True)


(    window_size  n_lags   lambda  r2_insample_stage1  r2_oos_stage1  \
 0           200       1  0.00010            0.761294      -1.384968   
 1           200       1  0.00001            0.890863      -5.522532   
 2           104       1  0.00010            0.955114      -1.341798   
 3           104       1  0.00001            0.994849      -1.916187   
 4            36       1  0.00001            0.980294      -0.817831   
 5            36       1  0.00010            0.976032      -0.804154   
 6           150       1  0.00010            0.883995      -1.745397   
 7           150       1  0.00001            0.992323      -4.854448   
 8            78       1  0.00010            0.973247      -1.044094   
 9            78       1  0.00001            0.993031      -1.279566   
 10           52       1  0.00001            0.987825      -0.886561   
 11           52       1  0.00010            0.979548      -0.819022   
 
     r2_insample_stage2  r2_oos_stage2         kappa   kappa_t

In [6]:
best_overall

window_size                   78.000000
n_lags                         1.000000
lambda                         0.000100
r2_insample_stage1             0.973247
r2_oos_stage1                 -1.044094
r2_insample_stage2             0.000042
r2_oos_stage2                 -0.004391
kappa                          0.004641
kappa_tstat                    0.274840
intercept                      0.000331
intercept_tstat                1.448967
n_observations              1806.000000
n_windows                   1808.000000
n_oos_predictions_stage2            NaN
objective                      0.000042
Name: 8, dtype: float64

In [ ]:
# import numpy as np
# import pandas as pd

results_all_rounds = []
# Initial grid
initial_param_grid = {
    
    # Evaluate objective
    summary_df['objective'] = summary_df.apply(objective_function, axis=1)

for iteration in range(5):

    # Run grid search
    summary_df, _ = grid_search(X, y, param_grid, verbose=True)

    # Compute objective
    summary_df["objective"] = summary_df.apply(objective_function, axis=1)
    results_all.append(summary_df)

    # Check for valid candidates
    valid_scores = summary_df.loc[summary_df["objective"] > -1.0, "objective"]
    if valid_scores.empty:
        break

    best_idx = valid_scores.idxmax()
    best_obj = valid_scores.max()

    # Convergence check
    if iteration > 0 and (best_obj - prev_best) < 1e-6:
        break

    prev_best = best_obj
    best = summary_df.loc[best_idx]

    # -------------------------------------------------
    # Refine grid around best point
    # -------------------------------------------------
    w = int(best["window_size"])
    l = int(best["n_lags"])
    lam = float(best["lambda"])

    param_grid = {
        "window_sizes": sorted(
            {int(w * f) for f in [0.75, 0.9, 1.0, 1.1, 1.25] if w * f >= 20}
        ),
        "n_lags": sorted({max(1, l - 2), l - 1, l, l + 1, l + 2}),
        "lambdas": lam * np.array([0.5, 0.75, 1.0, 1.25, 1.5]),
    }


print("\nBest hyperparameters after refinement:")
print(best_overall[['window_size', 'n_lags', 'lambda', 'objective']])

Best this iteration:
window_size    200.000000
n_lags          12.000000
lambda           0.000100
objective        0.000441
Name: 0, dtype: float64


SyntaxError: 'break' outside loop (4087755605.py, line 15)

In [10]:
# print for the 5 rows with the highest objective values r2 insample_stage2 and stage1, kappa ,kappa_tstat, lambda, window_size, n_lags
top_10 = final_df.nlargest(30, 'objective')
print(top_10[['r2_insample_stage2', 'r2_insample_stage1', 'r2_oos_stage2', 'kappa', 'kappa_tstat', 'lambda', 'window_size', 'n_lags']])

NameError: name 'final_df' is not defined

## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    # Search space
    window_size = trial.suggest_int("window_size", 10, 400, step=10)
    n_lags      = trial.suggest_int("n_lags", 1, 25)
    lam         = trial.suggest_float("lambda", 1e-5, 1e-2, log=True)

    # Run estimation
    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    # Primary objective: maximize in-sample R^2 from stage 2
    r2_2 = float(summary.get("r2_insample_stage2", np.nan))
    kappa = float(summary.get("kappa", np.nan))
    # define indicator for positive r2 insample stage 1
    r2_1 = float(summary.get("r2_insample_stage1", np.nan))
    positive_r2_1 = r2_1 > 0
    # define indicator for kappa t_stat greater 1.96
    kappa_tstat = float(summary.get("kappa_tstat", np.nan))
    kappa_significant = kappa_tstat > 1.96

    if not np.isfinite(r2_2):
        raise optuna.TrialPruned()

    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("r2_stage1", float(summary.get("r2_insample_stage1", np.nan)))
    trial.set_user_attr("kappa",     float(summary.get("kappa", np.nan)))
    trial.set_user_attr("kappa_tstat", float(summary.get("kappa_tstat", np.nan)))

    return r2_2 * kappa * positive_r2_1 * kappa_significant


sampler = TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=50,
    n_ei_candidates=64
)

study1 = optuna.create_study(direction="maximize", sampler=sampler)
study1.optimize(objective, n_trials=300, n_jobs=6, gc_after_trial=True)

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-01-28 07:50:34,647] A new study created in memory with name: no-name-91c10e75-27e8-4da6-95c1-4fb494423d0b
[I 2026-01-28 07:51:13,606] Trial 0 finished with value: -2.180994940204073e-09 and parameters: {'window_size': 140, 'n_lags': 4, 'lambda': 0.0015244386293093984}. Best is trial 0 with value: -2.180994940204073e-09.
[I 2026-01-28 07:51:45,887] Trial 2 finished with value: -1.704536511937249e-09 and parameters: {'window_size': 160, 'n_lags': 12, 'lambda': 0.003163567379362874}. Best is trial 2 with value: -1.704536511937249e-09.
[I 2026-01-28 07:51:46,234] Trial 3 finished with value: 0.0006257024969459346 and parameters: {'window_size': 370, 'n_lags': 7, 'lambda': 0.002138413627601193}. Best is trial 3 with value: 0.0006257024969459346.
[I 2026-01-28 07:52:22

In [ ]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "state", "params", "user_attrs", "system_attrs")
)

# save to csv
trials_df.to_csv("optuna_study_trials.csv", index=False)

In [10]:
print(trials_df.columns)

Index(['number', 'value', 'state', 'params_lambda', 'params_n_lags',
       'params_window_size', 'user_attrs_kappa', 'user_attrs_kappa_tstat',
       'user_attrs_r2_stage1', 'user_attrs_r2_stage2'],
      dtype='object')


In [11]:
# sort from highest R² (Optuna objective = "value") to lowest
df = trials_df.sort_values("value", ascending=False)

param_cols = [c for c in df.columns if c.startswith("params_")]

for _, row in df.iterrows():
    params = {c.replace("params_", ""): row[c] for c in param_cols}

    print(
        f"Trial {int(row['number'])}: "
        f"R²={row['value']:.6f}, "
        f"Params={params}, "
        f"kappa={row.get('user_attrs_kappa', float('nan')):.6g}, "
        f"t={row.get('user_attrs_kappa_tstat', float('nan')):.2f}, "
        f"R2_stage1={row.get('user_attrs_r2_stage1', float('nan')):.6f}, "
        f"R2_stage2={row.get('user_attrs_r2_stage2', float('nan')):.6f}"
    )


Trial 151: R²=0.003993, Params={'lambda': 0.00933750069285323, 'n_lags': 16, 'window_size': 40}, kappa=0.419003, t=4.66, R2_stage1=-0.006946, R2_stage2=0.003993
Trial 120: R²=0.003936, Params={'lambda': 0.005744194559514083, 'n_lags': 21, 'window_size': 120}, kappa=0.660438, t=7.73, R2_stage1=-0.001432, R2_stage2=0.003936
Trial 274: R²=0.003842, Params={'lambda': 0.0008452526710677865, 'n_lags': 16, 'window_size': 10}, kappa=0.057862, t=2.84, R2_stage1=0.850192, R2_stage2=0.003842
Trial 288: R²=0.003840, Params={'lambda': 0.0008080442333678375, 'n_lags': 16, 'window_size': 10}, kappa=0.0577157, t=2.83, R2_stage1=0.852952, R2_stage2=0.003840
Trial 259: R²=0.003840, Params={'lambda': 0.0008058768852209793, 'n_lags': 16, 'window_size': 10}, kappa=0.0577033, t=2.83, R2_stage1=0.853109, R2_stage2=0.003840
Trial 276: R²=0.003831, Params={'lambda': 0.0008283480784122526, 'n_lags': 16, 'window_size': 10}, kappa=0.0577202, t=2.83, R2_stage1=0.851461, R2_stage2=0.003831
Trial 178: R²=0.003778, P